In [16]:
import pandas as pd
import numpy as np
import ast # Required to parse stringified lists/dictionaries

# Load datasets
movies = pd.read_csv('movies.csv')
credits = pd.read_csv('credits.csv')

# Merge datasets based on movie title
movies = movies.merge(credits, on='title')

# Select only the relevant columns for our recommendation engine
# We need content-rich columns to build our 'tags'
movies = movies[['movie_id', 'title', 'overview', 'genres', 'keywords', 'cast', 'crew']]

# Drop rows with missing data (usually very few)
movies.dropna(inplace=True)

In [17]:
# Helper function to convert stringified lists of dictionaries into simple lists of names
def convert_to_list(obj):
    L = []
    for i in ast.literal_eval(obj):
        L.append(i['name'])
    return L

# Apply conversion to genres and keywords
movies['genres'] = movies['genres'].apply(convert_to_list)
movies['keywords'] = movies['keywords'].apply(convert_to_list)

# Get top 3 actors from the cast
def get_top_cast(obj):
    L = []
    counter = 0
    for i in ast.literal_eval(obj):
        if counter != 3:
            L.append(i['name'])
            counter += 1
        else:
            break
    return L

movies['cast'] = movies['cast'].apply(get_top_cast)

# Extract only the director's name from the crew column
def get_director(obj):
    L = []
    for i in ast.literal_eval(obj):
        if i['job'] == 'Director':
            L.append(i['name'])
            break
    return L

movies['crew'] = movies['crew'].apply(get_director)

In [18]:
# Remove spaces to create unique tags (e.g., 'Science Fiction' -> 'ScienceFiction')
def collapse_spaces(L):
    return [i.replace(" ","") for i in L]

movies['cast'] = movies['cast'].apply(collapse_spaces)
movies['crew'] = movies['crew'].apply(collapse_spaces)
movies['genres'] = movies['genres'].apply(collapse_spaces)
movies['keywords'] = movies['keywords'].apply(collapse_spaces)

# Convert 'overview' from string to list of words
movies['overview'] = movies['overview'].apply(lambda x: x.split())

# Create the final 'tags' column by merging everything
movies['tags'] = movies['overview'] + movies['genres'] + movies['keywords'] + movies['cast'] + movies['crew']

# Create a clean dataframe for the model
new_df = movies[['movie_id', 'title', 'tags']]

# Convert tags back into a single string (lowercase) for vectorization
new_df['tags'] = new_df['tags'].apply(lambda x: " ".join(x).lower())

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Initialize CountVectorizer to find the 5000 most frequent words, excluding English stop words
cv = CountVectorizer(max_features=5000, stop_words='english')

# Transform tags into vectors
vectors = cv.fit_transform(new_df['tags']).toarray()

# Calculate similarity scores between all movies (0 to 1 range)
similarity = cosine_similarity(vectors)

In [20]:
def recommend(movie):
    # Find the index of the movie in our dataframe
    movie_index = new_df[new_df['title'] == movie].index[0]
    
    # Get similarity scores for this movie and sort them (highest first)
    distances = similarity[movie_index]
    movies_list = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6]
    
    # Print the top 5 recommended movie titles
    print(f"Recommendations for '{movie}':")
    for i in movies_list:
        print(new_df.iloc[i[0]].title)

# TEST: Try it out with a movie!
recommend('Avatar')

Recommendations for 'Avatar':
Titan A.E.
Small Soldiers
Independence Day
Ender's Game
Aliens vs Predator: Requiem


In [21]:
import pickle

# Save the movie list (as a dictionary for easier loading)
pickle.dump(new_df.to_dict(), open('movie_dict.pkl', 'wb'))

# Save the similarity matrix
pickle.dump(similarity, open('similarity.pkl', 'wb'))